In [2]:
NOMBRE_APP = "NYC_Taxi_Data"  # Nombre de la spark session
DATASET_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet" 
RUTA_RAW = "./data/raw"  # Ruta del archivo raw
RUTA_TAXI_RAW = "./data/raw/yellow_tripdata_2024-01.parquet"
RUTA_PROCESSED = "./data/processed"  # Ruta del archivo procesado
RUTA_DB = "./data/nyc_taxi.db"  # Ruta de la base SQLite
SEMILLA = 42  # Semilla para la generación de números aleatorios
COLUMNAS_REQUERIDAS = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type", "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount"]  # Lista de columnas que el dataset debe contener para ser considerado válido.

CARGA_DATOS

In [ ]:
# COLUMNAS_REQUIERIDAS describe las columnas que se esperan en el dataset de NYC Taxi. 
# A continuación se detalla qué representa cada columna:
'''
Columna	                                        Qué es
tpep_pickup_datetime / tpep_dropoff_datetime	Inicio y fin del viaje
passenger_count	                                Pasajeros (lo carga el chofer, tiene errores)
trip_distance	                                Distancia en millas
PULocationID / DOLocationID	                    Zona de subida y bajada
RatecodeID	                                    Tipo de tarifa (1 normal, 2 aeropuerto JFK, etc.)
payment_type	                                1 tarjeta, 2 efectivo, 3 sin cargo, 4 disputa
fare_amount	                                    Tarifa base, sin extras
tip_amount, tolls_amount, etc.	                Propina, peajes y recargos
total_amount	                                Total cobrado
'''

In [15]:
from importlib.resources import path
import os
import requests

def descargar_dataset(url, ruta_destino): # ruta_destino = RUTA_RAW
    """
    Parámetros:
    url (str): La URL del dataset a descargar.
    ruta_destino (str): La ruta donde se guardará el archivo descargado.
    como spark solo lee archivos locales, primero se descarga el dataset y luego se lee desde la ruta local.
    """
    if os.path.exists(ruta_destino):
        print(f"El archivo ya existe en la ruta: {ruta_destino}. No se descargará nuevamente.")
        return ruta_destino  # Retorna la ruta del archivo existente
    
    # Creo la carpeta donde va a quedar el archivo
    os.makedirs(os.path.dirname(ruta_destino), exist_ok=True)

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()  # Lanza una excepción si la descarga falla
        with open(ruta_destino, 'wb') as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024): # Descarga en bloques de 1 MB
                file.write(chunk)
    print(f'MODULO CARGA_DATOS, FUNCION DESCARGAR_DATASET: Archivo descargado desde {url} y guardado en {ruta_destino}.')
    return ruta_destino  


def leer_dataset(spark, ruta_archivo): # ruta_archivo = RUTA_RAW
    df = spark.read.parquet(ruta_archivo)  # Lee el archivo parquet
    return df

def seleccionar_columnas(dataframe, columnas_requeridas): # columnas_requeridas = COLUMNAS_REQUERIDAS
    """
    Parámetros:
    dataframe (DataFrame): El DataFrame de Spark que se va a validar.
    columnas_requeridas (list): Una lista de nombres de columnas que se esperan en el DataFrame.
    
    Retorna:
    DataFrame: Un nuevo DataFrame que contiene solo las columnas requeridas.
    """
    print("Columnas antes:", len(dataframe.columns))
    dataframe = dataframe.select(columnas_requeridas)
    print("Columnas después:", len(dataframe.columns))
    return dataframe
    # en spark los df son inmutables, por lo que se retorna un nuevo df con las columnas requeridas

def cargar_datos(spark):
  
    # recibe la sesión como parámetro en lugar de crearla: la sesión (y el Driver) se crea una sola vez en main.py
    # Carga los datos desde un archivo parquet y los devuelve en un dataframe de spark
    
    df = spark.read.parquet(path, header=True, inferSchema=True) 
    
    # header = True indica que la primera fila del archivo parquet contiene los nombres de las columnas
    # inferSchema = True indica que spark debe inferir el tipo de dato de cada columna             
    return df

SESION_SPARK

In [5]:
from pyspark.sql import SparkSession

def crear_sesion(nombre_app):
    """
    Crea la sesion de spark y la retorna
    """
    sesion = (
        SparkSession.builder
        .appName(nombre_app)                    # nombre de la aplicacion
        .master("local[*]")                     # le dice a spark que corra localmente y que use los nucleos del procesador como executors
        .getOrCreate()                          # crear u obtiene la sesion de spark, si ya hay una sesion abierta la reutiliza, sino crea una nueva
    )                                           # dentro del parentesis = una sola instruccion / syntax error
    sesion.sparkContext.setLogLevel("ERROR")    # setea el nivel de log a ERROR para no mostrar INFO y WARN
    return sesion

MAIN

In [20]:
#from src.sesion_spark import crear_sesion
#from src import config
#from src.carga_datos import descargar_dataset, leer_dataset, validar_columnas
from pyspark.sql.functions import col, count, when
spark = crear_sesion(NOMBRE_APP)

ruta = descargar_dataset(DATASET_URL, RUTA_TAXI_RAW)
df = leer_dataset(spark, ruta)
df = seleccionar_columnas(df, COLUMNAS_REQUERIDAS)

print('*'*50)
df.show(5)
print('*'*50)
print("Filas:", df.count())
print("Tipos de datos:")
for col_name, col_type in df.dtypes:
    print(f"  {col_name}: {col_type}")
print('*'*50)
print('valores nulos por columna:')
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()
print('*'*50)
print('Estadisticas descriptivas del dataset:')
df.describe().show()




spark.stop()

El archivo ya existe en la ruta: ./data/raw/yellow_tripdata_2024-01.parquet. No se descargará nuevamente.
Columnas antes: 19
Columnas después: 17
**************************************************
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|      

In [ ]:
# dtypes del spark dataset (formato parquet)
'''
Tipo	                Qué es	                                        Columnas
************************************************************************************************************
int	                    Número entero (hasta unos ±2.100 millones)	    VendorID, PULocationID, DOLocationID
bigint	                Número entero “grande” (hasta ±9 trillones).    Es lo mismo que long en printSchema	passenger_count,        RatecodeID,                                     payment_type
double	                Número con decimales	                        distancias y todos los montos
string	                Texto	                                        tore_and_fwd_flag (“Y”/”N”)
timestamp_ntz	        Fecha y hora sin zona horaria 	                subida y bajada
'''

'\nTipo\t                Qué es\t                                        Columnas\nint\t                    Número entero (hasta unos ±2.100 millones)\t    VendorID, PULocationID, DOLocationID\nbigint\t                Número entero “grande” (hasta ±9 trillones).    Es lo mismo que long en printSchema\tpassenger_count,        RatecodeID,                                     payment_type\ndouble\t                Número con decimales\t                        distancias y todos los montos\nstring\t                Texto\t                                        tore_and_fwd_flag (“Y”/”N”)\ntimestamp_ntz\t        Fecha y hora sin zona horaria \t                subida y bajada\n'

# Dato para el equipo sobre las columnas 

'VendorID', 'RatecodeID', 'payment_type', 'PULocationID' y 'DOLocationID' son números, pero no son cantidades: son códigos o categorías. 

El 2 de payment_type (efectivo) no es “el doble” del 1 (tarjeta). Esto le importa al Integrante 3: en MLlib esas columnas no se pueden meter como números sin más, se tratan como categorías (por ejemplo, con OneHotEncoder). Y al Integrante 2: no tiene sentido calcular el “promedio de payment_type"

LIMPIEZA

In [ ]:
"""
Módulo: limpieza.py
Descripción: quita los datos erronesos y los nulos del dataset
Autor/a: <Josefina Mercado> — Legajo <número>
Proyecto: 2° Desempeño — Procesamiento de Datos (ISSD)

Entrada: <qué recibe el módulo>
Salida: <qué devuelve o qué archivo genera>
"""
from src.config import DICT_TIPOS
from pyspark.sql.functions import col

def corregir_tipos(df, tipos): 
    """
    Dtypes a corregir:
    bigint -> int (siempre que no haya valores mayores a 2.147.483.647)
    Al leerse desde Parquet, el esquema ya venía tipado correctamente. 
    Solo se convierten tres columnas de códigos de bigint a int, por consistencia y menor uso de memoria. 
    Las fechas se mantuvieron como timestamp_ntz para preservar la hora local de Nueva York.

    Parámetros: df (DataFrame) y tipos (dict: columna → tipo destino).
    Retorna: el DataFrame con los tipos corregidos.
    """
    for columna, tipo in tipos.items():
        df = df.withColumn(columna, col(columna).cast(tipo))
    return df


def eliminar_duplicados(dataframe): 
    """
    Parámetros:
    dataframe (DataFrame): El DataFrame de Spark que se va a limpiar.
  
    Retorna:
    DataFrame: Un DataFrame de Spark sin filas duplicadas.
    """
    # Implementación de la función para eliminar filas duplicadas del DataFrame
    pass  # Reemplazar con la lógica de eliminación

def tratar_nulos(dataframe): 
    """
    Parámetros:
    dataframe (DataFrame): El DataFrame de Spark que se va a limpiar.
  
    Retorna:
    DataFrame: Un DataFrame de Spark sin valores nulos.
    """
    # Implementación de la función para tratar los valores nulos en el DataFrame
    pass  # Reemplazar con la lógica de tratamiento de nulos

def filtrar_invalidos(df):
    """
    Parámetros:
    df (DataFrame): El DataFrame de Spark que se va a limpiar.
  
    Retorna:
    DataFrame: Un DataFrame de Spark sin filas con datos inválidos.
    """
    # Implementación de la función para filtrar filas con datos inválidos
    pass  # Reemplazar con la lógica de filtrado

def resumen_limpieza(dataframe): 
    """
    Parámetros:
    dataframe (DataFrame): El DataFrame de Spark que se va a limpiar.
  
    Retorna:
    dict: Un diccionario con un resumen de la limpieza realizada, incluyendo el número de filas eliminadas y el número de valores nulos tratados.
    """
    # Implementación de la función para generar un resumen de la limpieza
    pass  # Reemplazar con la lógica de resumen

def limpiar_dataset(dataframe): 
    """
    Parámetros:
    dataframe (DataFrame): El DataFrame de Spark que se va a limpiar.
  
    Retorna:
    DataFrame: Un DataFrame de Spark limpio y listo para procesar.
    """
    # Implementación de la función para limpiar el dataset completo
    pass  # Reemplazar con la lógica completa de limpieza